# 03 — Gold: Business Metrics

Reads from the Silver layer and computes pre-aggregated KPI collections for the Gold layer in MongoDB. Three KPI sets are produced: hourly trip volume by borough, revenue breakdown by pickup zone, and congestion fee impact analysis (2025+ data only).

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work')

from src.spark_utils import get_spark, read_mongo, write_mongo
from core.config.settings import settings
from core.audit.control_manager import ControlManager, ExecutionStatus
import pyspark.sql.functions as F

print("Imports OK.")

In [ ]:
spark = get_spark(app_name="TLC_Gold_Metrics")
control = ControlManager("gold_metrics")
execution = control.start({"source": "tlc_silver.trips_yellow"})

## Read Silver Data

In [ ]:
df = read_mongo(spark, settings.MONGO_DB_SILVER, "trips_yellow")
print(f"Silver records loaded: {df.count():,}")

## KPI 1 — Hourly Trip Volume by Borough

In [ ]:
kpi_hourly = (
    df
    .withColumn("pickup_hour",   F.hour("datetimes.pickup"))
    .withColumn("pickup_date",   F.to_date("datetimes.pickup"))
    .withColumn("pickup_borough", F.col("locations.pickup.borough"))
    .groupBy("pickup_date", "pickup_hour", "pickup_borough")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("financials.total_amount"), 2).alias("avg_total_amount"),
        F.round(F.sum("financials.total_amount"), 2).alias("total_revenue"),
    )
    .orderBy("pickup_date", "pickup_hour")
)

n1 = write_mongo(kpi_hourly, settings.MONGO_DB_GOLD, "kpi_hourly_volume", mode="overwrite")
print(f"kpi_hourly_volume: {n1:,} rows written")

## KPI 2 — Revenue by Pickup Zone

In [ ]:
kpi_zone = (
    df
    .groupBy(
        F.col("locations.pickup.zone_id").alias("zone_id"),
        F.col("locations.pickup.borough").alias("borough"),
        F.col("locations.pickup.zone").alias("zone_name"),
    )
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.sum("financials.fare_amount"), 2).alias("total_fare"),
        F.round(F.sum("financials.tip_amount"),  2).alias("total_tips"),
        F.round(F.sum("financials.total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("metrics.distance_miles"),  2).alias("avg_distance_miles"),
    )
    .orderBy("total_revenue", ascending=False)
)

n2 = write_mongo(kpi_zone, settings.MONGO_DB_GOLD, "kpi_revenue_zone", mode="overwrite")
print(f"kpi_revenue_zone: {n2:,} rows written")

## KPI 3 — Congestion Fee Impact (2025+)

In [ ]:
# cbd_congestion_fee is only populated for 2025+ records
df_2025 = df.filter(F.col("_meta.source_year") >= 2025)
n_2025  = df_2025.count()

if n_2025 > 0:
    kpi_congestion = (
        df_2025
        .withColumn(
            "paid_congestion_fee",
            F.when(F.col("financials.cbd_congestion_fee") > 0, True).otherwise(False),
        )
        .groupBy("paid_congestion_fee", F.col("locations.pickup.borough").alias("borough"))
        .agg(
            F.count("*").alias("trip_count"),
            F.round(F.sum("financials.cbd_congestion_fee"), 2).alias("total_congestion_fees"),
            F.round(F.avg("financials.total_amount"), 2).alias("avg_total_amount"),
        )
    )
    n3 = write_mongo(kpi_congestion, settings.MONGO_DB_GOLD, "kpi_congestion_impact", mode="overwrite")
    print(f"kpi_congestion_impact: {n3:,} rows written (from {n_2025:,} 2025+ records)")
else:
    print("No 2025+ records found. kpi_congestion_impact skipped.")
    n3 = 0

In [ ]:
control.end(
    ExecutionStatus.SUCCESS,
    {
        "kpi_hourly_volume":    n1,
        "kpi_revenue_zone":     n2,
        "kpi_congestion_impact": n3,
    },
)
import json
print(json.dumps(control.get_report(), indent=2, default=str))